In [1]:
import pandas as pd
project_path = r"C:\Users\Bawan Singh\OneDrive\Documents\Arshleen\market-portfolio-risk-analysis"

returns = pd.read_csv(
    project_path + r"\data\processed\daily_returns.csv",
    index_col=0,
    parse_dates=True
)

In [2]:
returns.head()

,GLD,QQQ,SPY,TLT
Date,,,,
2019-01-03,0.009066,-0.032671,-0.023863,0.011379
2019-01-04,-0.008086,0.042785,0.033496,-0.011575
2019-01-07,0.003458,0.011906,0.007885,-0.002948
2019-01-08,-0.002708,0.009045,0.009395,-0.002629
2019-01-09,0.006418,0.008149,0.004673,-0.001564


In [3]:
weights = {
    "Conservative": {
        "SPY": 0.40,
        "TLT": 0.40,
        "GLD": 0.20
    },
    "Balanced": {
        "SPY": 0.60,
        "TLT": 0.30,
        "GLD": 0.10
    },
    "Growth": {
        "SPY": 0.40,
        "QQQ": 0.50,
        "TLT": 0.10
    }
}

weights

{'Conservative': {'SPY': 0.4, 'TLT': 0.4, 'GLD': 0.2},
 'Balanced': {'SPY': 0.6, 'TLT': 0.3, 'GLD': 0.1},
 'Growth': {'SPY': 0.4, 'QQQ': 0.5, 'TLT': 0.1}}

In [4]:
for portfolio, assets in weights.items():
    print(portfolio, sum(assets.values()))

Conservative 1.0
Balanced 0.9999999999999999
Growth 1.0


In [5]:
import numpy as np
np.isclose(sum(weights["Balanced"].values()), 1)


True

In [6]:
conservative_returns = (
    returns["SPY"] * 0.40
    + returns["TLT"] * 0.40
    + returns["GLD"] * 0.20
)

conservative_returns.head()

Date
2019-01-03   -0.003180
2019-01-04    0.007151
2019-01-07    0.002666
2019-01-08    0.002165
2019-01-09    0.002527
dtype: float64

In [7]:
portfolio_returns = pd.DataFrame()

portfolio_returns["Conservative"] = (
    returns["SPY"] * weights["Conservative"]["SPY"]
    + returns["TLT"] * weights["Conservative"]["TLT"]
    + returns["GLD"] * weights["Conservative"]["GLD"]
)

portfolio_returns["Balanced"] = (
    returns["SPY"] * weights["Balanced"]["SPY"]
    + returns["TLT"] * weights["Balanced"]["TLT"]
    + returns["GLD"] * weights["Balanced"]["GLD"]
)

portfolio_returns["Growth"] = (
    returns["SPY"] * weights["Growth"]["SPY"]
    + returns["QQQ"] * weights["Growth"]["QQQ"]
    + returns["TLT"] * weights["Growth"]["TLT"]
)

portfolio_returns.head()

,Conservative,Balanced,Growth
Date,,,
2019-01-03,-0.003180,-0.009997,-0.024742
2019-01-04,0.007151,0.015816,0.033633
2019-01-07,0.002666,0.004192,0.008812
2019-01-08,0.002165,0.004578,0.008018
2019-01-09,0.002527,0.002977,0.005788


In [8]:
portfolio_summary=pd.DataFrame({"Average Daily Return":portfolio_returns.mean(), "Daily Volatility":portfolio_returns.std()})
portfolio_summary

,Average Daily Return,Daily Volatility
Conservative,0.000417,0.006810
Balanced,0.000489,0.007940
Growth,0.000743,0.012346


In [9]:
portfolio_summary_percent = portfolio_summary * 100
portfolio_summary_percent.style.format("{:.2f}%")


,Average Daily Return,Daily Volatility
Conservative,0.04%,0.68%
Balanced,0.05%,0.79%
Growth,0.07%,1.23%


In [10]:
portfolio_annualized_return = portfolio_returns.mean() * 252
portfolio_annualized_volatility = portfolio_returns.std() * np.sqrt(252)

portfolio_annualized_summary = pd.DataFrame({
    "Annualized Return": portfolio_annualized_return,
    "Annualized Volatility": portfolio_annualized_volatility
})

portfolio_annualized_summary.style.format("{:.2%}")

,Annualized Return,Annualized Volatility
Conservative,10.50%,10.81%
Balanced,12.32%,12.60%
Growth,18.73%,19.60%


In [21]:
portfolio_annualized_return.to_csv(
    project_path + r"\data\processed\portfolio_annualized_return.csv"
)

In [22]:
portfolio_annualized_volatility = pd.read_csv(
    project_path + r"\data\processed\portfolio_annualized_volatility.csv",
    index_col=0
).squeeze()

In [12]:
sharpe_ratio = (
    portfolio_annualized_return
    / portfolio_annualized_volatility
)

sharpe_ratio

Conservative    0.971717
Balanced        0.977689
Growth          0.955622
dtype: float64

In [13]:
sharpe_ratio.to_csv(
    project_path + r"\data\processed\sharpe_ratio.csv"
)

In [14]:
sharpe_ratio_table = pd.DataFrame({
    "Sharpe Ratio": sharpe_ratio
})

sharpe_ratio_table.style.format("{:.2f}")

,Sharpe Ratio
Conservative,0.97
Balanced,0.98
Growth,0.96


In [15]:
downside_returns = portfolio_returns.clip(upper=0)

downside_deviation = downside_returns.std() * np.sqrt(252)

sortino_ratio = (
    portfolio_annualized_return
    / downside_deviation
)

sortino_ratio_table = pd.DataFrame({
    "Sortino Ratio": sortino_ratio
})

sortino_ratio_table.style.format("{:.2f}")

,Sortino Ratio
Conservative,1.59
Balanced,1.56
Growth,1.52


In [16]:
cumulative_returns = (1 + portfolio_returns).cumprod()

cumulative_returns.head()

,Conservative,Balanced,Growth
Date,,,
2019-01-03,0.996820,0.990003,0.975258
2019-01-04,1.003948,1.005661,1.008059
2019-01-07,1.006625,1.009877,1.016941
2019-01-08,1.008804,1.014500,1.025095
2019-01-09,1.011354,1.017520,1.031028


In [17]:
running_peak = cumulative_returns.cummax()

drawdown = (cumulative_returns - running_peak) / running_peak

drawdown.head()
##drawdown is measured from the previous peak, not from the original investment.

,Conservative,Balanced,Growth
Date,,,
2019-01-03,0.0,0.0,0.0
2019-01-04,0.0,0.0,0.0
2019-01-07,0.0,0.0,0.0
2019-01-08,0.0,0.0,0.0
2019-01-09,0.0,0.0,0.0


In [18]:
max_drawdown = drawdown.min()

max_drawdown

Conservative   -0.250688
Balanced       -0.248956
Growth         -0.303071
dtype: float64

In [19]:
max_drawdown_table=pd.DataFrame({"Maximum Drawdown":max_drawdown})
max_drawdown_table.style.format("{:.2%}")

,Maximum Drawdown
Conservative,-25.07%
Balanced,-24.90%
Growth,-30.31%


In [20]:
portfolio_returns.to_csv(
    project_path + r"\data\processed\portfolio_returns.csv"
)

In [23]:
sharpe_ratio = pd.read_csv(
    project_path + r"\data\processed\sharpe_ratio.csv",
    index_col=0
).squeeze()

In [24]:
sharpe_ratio

Conservative    0.971717
Balanced        0.977689
Growth          0.955622
Name: 0, dtype: float64

In [25]:
performance_summary["Sharpe Ratio"] = sharpe_ratio

NameError: name 'performance_summary' is not defined